In [1]:
import sys, os, glob as _g, subprocess
try:
    import onnxruntime; print(f'ort {onnxruntime.__version__} ready')
except ImportError:
    wdirs = {os.path.dirname(w) for w in _g.glob('/kaggle/input/**/*.whl', recursive=True)
             if 'onnxruntime' in os.path.basename(w)}
    if not wdirs: raise RuntimeError('no onnxruntime wheel in /kaggle/input')
    fl = [x for d in wdirs for x in ['--find-links', d]]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', *fl, 'onnxruntime'], check=True)
    import onnxruntime; print(f'ort {onnxruntime.__version__} installed')


ort 1.24.4 installed


# BirdCLEF 2026 Pseudo-Label Generation v34

## Purpose
Run v30 GRU inference on **test soundscapes**, save:
1. Per-window Perch embeddings as `test_{sc_stem}_{end_secs}s.npy`
2. Per-window soft label predictions as `pseudo_labels_v34.pkl`

## Output datasets
- `birdclef-2026-test-perch-embs-v34` — test embeddings (1536-d per window)
- `birdclef-2026-pseudo-labels-v34` — soft label dict for training

## Required datasets
1. `birdclef-2026` (test_soundscapes, sample_submission.csv, taxonomy.csv)
2. `chiragggg/birdclef-2026-perch-onnx`
3. `chiragggg/birdclef-2026-perch-weights-v30`


In [ ]:
# === CELL 2: IMPORTS & CONFIG ===
import os, warnings, gc, pickle
from pathlib import Path
import numpy as np, pandas as pd, soundfile as sf, librosa, onnxruntime as ort
import torch, torch.nn as nn
from torch.cuda.amp import autocast
from tqdm import tqdm
warnings.filterwarnings('ignore')

CFG = dict(
    folds=5,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    perch_sr=32000, perch_seconds=5, perch_emb_dim=1536, perch_batch=16,
    gru_hidden=512,
    gru_layers=2,
    gauss_sigma=0.0,          # no smoothing for pseudo labels
    min_window_conf=0.2,      # min max-prob per window to include in training
)
CFG['perch_target'] = CFG['perch_sr'] * CFG['perch_seconds']  # 160000
device = torch.device(CFG['device'])
torch.set_num_threads(os.cpu_count() or 4)
print(f'Device: {device}')
print(f'min_window_conf={CFG["min_window_conf"]}  (windows below this are skipped)')


In [ ]:
# === CELL 3: PATHS & SPECIES ===
def _fe(*c):
    return next((p for p in c if os.path.exists(p)), c[0])

TAXONOMY_CSV = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                   '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
TEST_AUDIO   = _fe('/kaggle/input/birdclef-2026/test_soundscapes',
                   '/kaggle/input/competitions/birdclef-2026/test_soundscapes')
SAMPLE_SUB   = _fe('/kaggle/input/birdclef-2026/sample_submission.csv',
                   '/kaggle/input/competitions/birdclef-2026/sample_submission.csv')
GRU_CKPT_DIR = _fe('/kaggle/input/birdclef-2026-perch-weights-v30',
                   '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v30',
                   '/kaggle/working')
ONNX_PATH = None
for _c in [
    '/kaggle/input/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx',
]:
    if os.path.exists(_c):
        ONNX_PATH = _c
        break

OUT_EMB_DIR  = '/kaggle/working/test_embs'
OUT_LABEL_DIR = '/kaggle/working'
os.makedirs(OUT_EMB_DIR, exist_ok=True)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
n_classes   = len(species)
print(f'Species      : {n_classes}')
print(f'GRU_CKPT_DIR : {GRU_CKPT_DIR}')
print(f'ONNX_PATH    : {ONNX_PATH}')
print(f'OUT_EMB_DIR  : {OUT_EMB_DIR}')


In [ ]:
# === CELL 4: PERCHGRU MODEL (v30 architecture) ===
class PerchGRU(nn.Module):
    def __init__(self, n_classes, emb_dim=1536, hidden=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim), nn.Linear(emb_dim, 512), nn.GELU(),
        )
        self.gru = nn.GRU(512, hidden, n_layers, batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2), nn.Dropout(0.2), nn.Linear(hidden * 2, n_classes),
        )
    def forward(self, x):
        single = (x.dim() == 2)
        if single: x = x.unsqueeze(1)
        h, _ = self.gru(self.proj(x))
        out   = self.head(h)
        return out.squeeze(1) if single else out

print('PerchGRU defined')


In [ ]:
# === CELL 5: LOAD V30 CHECKPOINTS ===
def _load_gru(names, ckpt_dir):
    ms = []
    for n in names:
        p = Path(ckpt_dir) / n
        if not p.exists():
            print(f'  MISSING: {p}')
            continue
        m = PerchGRU(n_classes, emb_dim=CFG['perch_emb_dim'],
                     hidden=CFG['gru_hidden'], n_layers=CFG['gru_layers']).to(device)
        m.load_state_dict(torch.load(p, map_location=device, weights_only=True))
        m.eval()
        ms.append(m)
        print(f'  OK {n}')
    return ms

print('Loading v30 GRU checkpoints...')
gru_models = _load_gru(
    [f'perch_gru_v30_fold{i}.pt' for i in range(CFG['folds'])],
    GRU_CKPT_DIR,
)
print(f'Loaded: {len(gru_models)}/5')


In [ ]:
# === CELL 6: INIT PERCH ONNX ===
_sess = None; _inp = None; _eidx = 0; _onnx_ok = False
if ONNX_PATH is None:
    print('ONNX not found')
else:
    try:
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        opts.intra_op_num_threads = os.cpu_count() or 4
        _sess = ort.InferenceSession(ONNX_PATH, sess_options=opts,
                                     providers=['CPUExecutionProvider'])
        _inp  = _sess.get_inputs()[0].name
        _out_names = [o.name for o in _sess.get_outputs()]
        ekey  = next((o.name for o in _sess.get_outputs()
                      if o.shape and o.shape[-1] == 1536), _out_names[0])
        _eidx = _out_names.index(ekey)
        _t    = _sess.run(None, {_inp: np.zeros((1, CFG['perch_target']), np.float32)})
        _e    = _t[_eidx]
        if _e.ndim == 3: _e = _e.mean(1)
        assert _e.shape[-1] == 1536, f'Expected 1536-d emb, got {_e.shape}'
        _onnx_ok = True
        print(f'ONNX OK: {Path(ONNX_PATH).name}  emb={_e.shape}')
    except Exception as ex:
        print(f'ONNX ERROR: {ex}')


In [ ]:
# === CELL 7: GENERATE PSEUDO LABELS FOR TEST SOUNDSCAPES ===
_amp = (device.type == 'cuda')

sub = pd.read_csv(SAMPLE_SUB).copy()
sub['_sc'] = sub['row_id'].str.rsplit('_', n=1).str[0]
print(f'Test rows: {len(sub)}  soundscapes: {sub["_sc"].nunique()}')

# Result: list of seq_groups in training format
# {stem: sc_stem, windows: [(emb_stem, end_secs, soft_label_vector), ...]}
pseudo_groups = []
n_windows_kept = 0
n_windows_total = 0

for sc, grp in tqdm(sub.groupby('_sc'), desc='soundscapes'):
    rids = [str(r) for r in grp['row_id']]
    ap = None
    for ext in ['.ogg', '.wav', '.flac']:
        c = Path(TEST_AUDIO) / f'{sc}{ext}'
        if c.exists(): ap = str(c); break
    if ap is None: continue

    ends = []
    for r in rids:
        try: ends.append(int(r.rsplit('_', 1)[-1]))
        except: pass
    if not ends: continue

    # Read + resample audio
    try:
        y, sr = sf.read(ap, always_2d=False)
        if y.ndim == 2: y = y.mean(1)
        if sr != CFG['perch_sr']:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=CFG['perch_sr'])
        y = y.astype(np.float32)
    except Exception as e:
        print(f'[W] read {sc}: {e}'); continue

    # Embed all windows with Perch ONNX
    clips = []
    for es in ends:
        e0 = int(es * CFG['perch_sr'])
        s0 = max(0, e0 - CFG['perch_target'])
        c  = y[s0:e0]
        if len(c) < CFG['perch_target']:
            c = np.pad(c, (0, CFG['perch_target'] - len(c)))
        clips.append(c)

    all_embs = []
    for bi in range(0, len(clips), CFG['perch_batch']):
        B   = np.stack(clips[bi:bi + CFG['perch_batch']])
        out = _sess.run(None, {_inp: B})[_eidx]
        if out.ndim == 3: out = out.mean(1)
        all_embs.append(out.astype(np.float32))
    emb_matrix = np.vstack(all_embs)  # (T, 1536)

    # Run v30 GRU on full sequence to get soft label predictions
    seq = torch.from_numpy(emb_matrix).float().unsqueeze(0).to(device)  # (1, T, 1536)
    fold_preds = []
    for m in gru_models:
        with torch.inference_mode(), autocast(enabled=_amp):
            fold_preds.append(torch.sigmoid(m(seq).float())[0].cpu().numpy())
    preds = np.mean(fold_preds, axis=0)  # (T, n_classes)

    # Save embeddings and build pseudo windows
    sc_stem = sc  # use the sc identifier directly
    windows = []
    for t, (end_secs, pred) in enumerate(zip(ends, preds)):
        n_windows_total += 1
        # Confidence filter: skip windows with no confident prediction
        if pred.max() < CFG['min_window_conf']: continue

        # Save embedding
        emb_stem = f'test_{sc_stem}_{end_secs}s'
        np.save(os.path.join(OUT_EMB_DIR, emb_stem + '.npy'), emb_matrix[t])

        # Store window entry with soft label
        windows.append((emb_stem, end_secs, pred.astype('float32')))
        n_windows_kept += 1

    if windows:
        pseudo_groups.append({'stem': sc_stem, 'windows': windows})

print(f'\nDone.')
print(f'  Soundscapes with pseudo labels : {len(pseudo_groups)}')
print(f'  Windows kept / total           : {n_windows_kept} / {n_windows_total}')
print(f'  Keep rate                      : {100 * n_windows_kept / max(n_windows_total, 1):.1f}%')


In [ ]:
# === CELL 8: SAVE PSEUDO LABELS ===
PSEUDO_LABEL_PATH = os.path.join(OUT_LABEL_DIR, 'pseudo_labels_v34.pkl')
with open(PSEUDO_LABEL_PATH, 'wb') as f:
    pickle.dump(pseudo_groups, f)

print(f'Saved pseudo_labels_v34.pkl  ({len(pseudo_groups)} soundscapes)')
print(f'Embeddings saved to: {OUT_EMB_DIR}')
print(f'  Files: {len(list(Path(OUT_EMB_DIR).glob("*.npy")))}')
print('\nNext: upload both as Kaggle datasets:')
print('  birdclef-2026-test-perch-embs-v34  <- test_embs/ folder')
print('  birdclef-2026-pseudo-labels-v34    <- pseudo_labels_v34.pkl')


In [ ]:
# === CELL 9: UPLOAD DATASETS ===
import shutil, subprocess, json as _json

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'chiragggg')

# Upload 1: test embeddings
EMB_SLUG = 'birdclef-2026-test-perch-embs-v34'
_emb_meta = {'title': EMB_SLUG, 'id': f'{KAGGLE_USERNAME}/{EMB_SLUG}',
             'licenses': [{'name': 'CC0-1.0'}]}
with open(os.path.join(OUT_EMB_DIR, 'dataset-metadata.json'), 'w') as mf:
    _json.dump(_emb_meta, mf, indent=2)
r1 = subprocess.run(['kaggle', 'datasets', 'create', '-p', OUT_EMB_DIR, '--dir-mode', 'zip'],
                    capture_output=True, text=True)
print(r1.stdout)
if r1.returncode != 0: print('STDERR:', r1.stderr)

# Upload 2: pseudo labels pickle
LBL_SLUG = 'birdclef-2026-pseudo-labels-v34'
_lbl_dir = '/kaggle/working/upload_pseudo_labels'
os.makedirs(_lbl_dir, exist_ok=True)
shutil.copy2(PSEUDO_LABEL_PATH, _lbl_dir)
_lbl_meta = {'title': LBL_SLUG, 'id': f'{KAGGLE_USERNAME}/{LBL_SLUG}',
              'licenses': [{'name': 'CC0-1.0'}]}
with open(os.path.join(_lbl_dir, 'dataset-metadata.json'), 'w') as mf:
    _json.dump(_lbl_meta, mf, indent=2)
r2 = subprocess.run(['kaggle', 'datasets', 'create', '-p', _lbl_dir, '--dir-mode', 'zip'],
                    capture_output=True, text=True)
print(r2.stdout)
if r2.returncode != 0: print('STDERR:', r2.stderr)
else: print('Both datasets uploaded successfully.')
